In [32]:
import torchvision.models as visionmodels
import torch.nn as nn
import torch.optim as optim
import torch

class Emolens(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = visionmodels.efficientnet_b4(weights=visionmodels.EfficientNet_B4_Weights.DEFAULT)

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        for parameter in self.model.features[-1].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-2].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-3].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-4].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-5].parameters():
            parameter.requires_grad = True
        for parameter in self.model.features[-6].parameters():
            parameter.requires_grad = True


        self.model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1792, 7))

    def forward(self, x):
        x = self.model(x)

        return x

In [33]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [34]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('mv kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d mstjebashazida/affectnet -p /content/ --unzip')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


0

In [35]:
!mv 'archive (3)' data
!ls data

mv: cannot stat 'archive (3)': No such file or directory
'archive (3)'   labels.csv   Test   Train


In [36]:
!mv 'data/Test/Anger' 'data/Test/anger'
!mv 'data/Test/Contempt' 'data/Test/contempt'

mv: cannot stat 'data/Test/Anger': No such file or directory
mv: cannot stat 'data/Test/Contempt': No such file or directory


In [37]:
!rm -r 'data/Test/contempt'
!rm -r 'data/Train/contempt'

rm: cannot remove 'data/Test/contempt': No such file or directory
rm: cannot remove 'data/Train/contempt': No such file or directory


In [38]:
train_Data = "data/Train"
Test_Data = "data/Test"
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(root=train_Data, transform=train_transform)
test_dataset = datasets.ImageFolder(root=Test_Data, transform=test_transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [39]:
model = Emolens()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(model_parameters, lr=0.00005)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # monitor loss -- lower is better
    patience=2,       # wait 2 epochs before reducing
    factor=0.5        # reduce lr by half when triggered
)

In [40]:
print(train_dataset.class_to_idx)
print(test_dataset.class_to_idx)

{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


In [41]:
num_of_epochs=10

for epoch in range(num_of_epochs):
    sum_of_train_losses = 0
    sum_of_test_losses = 0
    total_correct = 0
    total_images = 0

    model.train()
    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        y_pred = model(images)
        loss = criterion(y_pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_of_train_losses += loss.item()

    model.eval()
    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)

            y_pred = model(images)
            loss = criterion(y_pred, labels)

            y_pred = torch.argmax(y_pred, dim=1)
            total_correct += (y_pred == labels).sum().item()
            total_images += labels.size(0)

            sum_of_test_losses += loss.item()

    accuracy = (total_correct / total_images) * 100
    print(f'Epoch {epoch+1}: Train Loss: {sum_of_train_losses / len(train_dataloader): .3f} | Test Loss: {sum_of_test_losses / len(test_dataloader): .3f} | Accuracy: {accuracy: .2f}%')

Epoch 1: Train Loss:  1.582 | Test Loss:  1.374 | Accuracy:  46.86%
Epoch 2: Train Loss:  1.079 | Test Loss:  1.227 | Accuracy:  55.97%
Epoch 3: Train Loss:  0.953 | Test Loss:  1.202 | Accuracy:  60.95%
Epoch 4: Train Loss:  0.874 | Test Loss:  1.113 | Accuracy:  63.00%
Epoch 5: Train Loss:  0.806 | Test Loss:  1.148 | Accuracy:  65.00%
Epoch 6: Train Loss:  0.746 | Test Loss:  1.112 | Accuracy:  65.44%
Epoch 7: Train Loss:  0.694 | Test Loss:  1.114 | Accuracy:  67.20%
Epoch 8: Train Loss:  0.643 | Test Loss:  1.168 | Accuracy:  68.02%
Epoch 9: Train Loss:  0.604 | Test Loss:  1.176 | Accuracy:  68.73%
Epoch 10: Train Loss:  0.560 | Test Loss:  1.180 | Accuracy:  68.31%


In [43]:
torch.save(model.state_dict(), 'model.pth')

In [44]:
!ls /content

affectnet.zip  data  drive  model.pth  sample_data


In [45]:
from google.colab import files
files.download('/content/model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>